In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [ ]:
# let's load some data
db = SQLDatabase.from_uri("sqlite:///../data/music.db")

In [ ]:
# this SQLDatabase object has some nice methods to work with the database
# they will be used by our agent
db.get_usable_table_names()

In [ ]:
db.run("SELECT * FROM Artist LIMIT 5;")

In [ ]:
# let's init our LLM
model = init_chat_model("openai:gpt-4o-mini")

In [ ]:

# with langchain is super easy to create a SQL toolkit to interact with sql data
toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

In [ ]:
# let's now create a nice agent
dialect = ""
top_k = 5

system_prompt = f"""
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
"""


In [ ]:
# let's build our agentzzz

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
)

In [ ]:
question = "Which genre on average has the longest tracks?"

responses = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

responses["messages"][-1].pretty_print()

In [ ]:
question = "List the 5 artists with the most albums."

responses = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)

responses["messages"][-1].pretty_print()